4 Data Cleaning & Preprocessing
4.1 Drop Unnecessary Columns
Prompt: Which columns should be dropped, and why?

In your Quarto report, explain:
- Which columns are irrelevant or redundant?
- Why are we removing multiple versions of NAICS/SOC codes?
- How will this improve analysis?

4.2 Dropping Unnecessary Columns
We remove redundant NAICS/SOC codes and tracking data to simplify our dataset. Keeping only the latest NAICS_2022_6 and SOC_2021_4 ensures that our analysis reflects current industry and occupational classifications.

4.2.1 Python Implementation
columns_to_drop = [
    "ID", "URL", "ACTIVE_URLS", "DUPLICATES", "LAST_UPDATED_TIMESTAMP",
    "NAICS2", "NAICS3", "NAICS4", "NAICS5", "NAICS6",
    "SOC_2", "SOC_3", "SOC_5"
]
df.drop(columns=columns_to_drop, inplace=True)

4.3 Handle Missing Values
While performing the analysis answer the question: How should missing values be handled?

📄 Quarto Markdown Example

4.4 Handling Missing Values
We use different strategies for missing values:
- Numerical fields (e.g., Salary) are filled with the median.
- Categorical fields (e.g., Industry) are replaced with "Unknown".
- Columns with >50% missing values are dropped.

import missingno as msno

# Visualize missing data
msno.heatmap(df)
plt.title("Missing Values Heatmap")
plt.show()

# Drop columns with >50% missing values
df.dropna(thresh=len(df) * 0.5, axis=1, inplace=True)

# Fill missing values
df["Salary"].fillna(df["Salary"].median(), inplace=True)
df["Industry"].fillna("Unknown", inplace=True)

4.5 Remove Duplicates
📄 Quarto Markdown Example
## Removing Duplicate Job Postings

To ensure each job is counted only once, we remove duplicates based on job title, company, location, and posting date.

df = df.drop_duplicates(subset=["TITLE", "COMPANY", "LOCATION", "POSTED"], keep="first")

5 Exploratory Data Analysis (EDA)
In your Quarto report, explain:
- Why these visualizations were chosen.
- Key insights from each graph.

📄 Quarto Markdown Example

5.1 Job Postings by Industry
Understanding industry demand helps job seekers prioritize skill development.

5.1.1 Job Postings by Industry
fig = px.bar(df["Industry"].value_counts(), title="Job Postings by Industry")
fig.show()

5.1.2 Salary Distribution by Industry
fig = px.box(df, x="Industry", y="Salary", title="Salary Distribution by Industry")
fig.show()

5.1.3 Remote vs. On-Site Jobs
fig = px.pie(df, names="REMOTE_TYPE_NAME", title="Remote vs. On-Site Jobs")
fig.show()

6 Commit & Publish to GitHub
Ensure _quarto.yml is updated to include the “Data Analysis” tab.

quarto render
git add data_analysis.qmd _quarto.yml
git commit -m "Enhanced data cleaning, EDA, and salary outlier handling"
git push origin main

## **Data Cleaning**

In [29]:
from pyspark.sql import SparkSession

# Start a Spark session
spark = SparkSession.builder.appName("JobPostingsAnalysis").getOrCreate()

# Load the CSV file into a Spark DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine","true").option("escape", "\"").csv("/home/ubuntu/job-market-analysis-project-2026/data/lightcast_job_postings.csv")

# Show schema
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- LAST_UPDATED_DATE: string (nullable = true)
 |-- LAST_UPDATED_TIMESTAMP: timestamp (nullable = true)
 |-- DUPLICATES: integer (nullable = true)
 |-- POSTED: string (nullable = true)
 |-- EXPIRED: string (nullable = true)
 |-- DURATION: integer (nullable = true)
 |-- SOURCE_TYPES: string (nullable = true)
 |-- SOURCES: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- ACTIVE_URLS: string (nullable = true)
 |-- ACTIVE_SOURCES_INFO: string (nullable = true)
 |-- TITLE_RAW: string (nullable = true)
 |-- BODY: string (nullable = true)
 |-- MODELED_EXPIRED: string (nullable = true)
 |-- MODELED_DURATION: integer (nullable = true)
 |-- COMPANY: integer (nullable = true)
 |-- COMPANY_NAME: string (nullable = true)
 |-- COMPANY_RAW: string (nullable = true)
 |-- COMPANY_IS_STAFFING: boolean (nullable = true)
 |-- EDUCATION_LEVELS: string (nullable = true)
 |-- EDUCATION_LEVELS_NAME: string (nullable = true)
 |-- MIN_EDULEVELS: integer (

**Cleaning the Dataset**

In [30]:
# select the needed columns 
cols = [
    # job roles/titles
    "TITLE_NAME",
    "TITLE_CLEAN",
    # skills/demaand
    "SKILLS_NAME",
    "SPECIALIZED_SKILLS_NAME",
    "SOFTWARE_SKILLS_NAME",
    "COMMON_SKILLS_NAME",
    "CERTIFICATION_NAMES",
    # job description
    "BODY",
    # industry
    "NAICS2_NAME",
    "NAICS3_NAME",
    "LIGHTCAST_SECTORS_NAME",
    LOT_CAREER_AREA_NAME:
    LOT_OCCUPATION_NAME: string (nullable = true)

    # experience/education
    "MIN_YEARS_EXPERIENCE",
    "MIN_EDULEVELS_NAME",
    "IS_INTERNSHIP",
    # salary
    "SALARY",
    "SALARY_FROM",
    "SALARY_TO",
    # work setup
    "REMOTE_TYPE_NAME",
    "EMPLOYMENT_TYPE_NAME"
]

df = df.select(cols)

In [25]:
# compute the percentage of missing data to assess the approach for data cleaning
from pyspark.sql.functions import col, count, when

total_rows = df.count()

missing_summary = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas().T

missing_summary.columns = ["missing_count"]
missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / total_rows * 100
)

missing_summary = missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

missing_summary

,missing_count,missing_percentage
LIGHTCAST_SECTORS_NAME,54711,75.465530
SALARY,41690,57.505035
SALARY_FROM,40100,55.311871
SALARY_TO,40100,55.311871
MIN_YEARS_EXPERIENCE,23146,31.926398
TITLE_CLEAN,140,0.193109
TITLE_NAME,44,0.060691
SPECIALIZED_SKILLS_NAME,44,0.060691
SKILLS_NAME,44,0.060691
NAICS3_NAME,44,0.060691


In [16]:
df.select("TITLE_NAME", "TITLE_CLEAN") \
  .distinct() \
  .show(50, truncate=False)

+-----------------------------------+-----------------------------------------------------------------------+
|TITLE_NAME                         |TITLE_CLEAN                                                            |
+-----------------------------------+-----------------------------------------------------------------------+
|Schedule Analysts                  |d f o analyst hybrid schedule                                          |
|SAP HR Consultants                 |sap hr consultant                                                      |
|Data Reporting Analysts            |oracle ta reporting analyst                                            |
|Business Intelligence Analysts     |data modernization bi analyst state and local                          |
|Practice Consultants               |associate consultant better practice solutions hybrid flex             |
|Research and Data Analysts         |data analyst ii research and data analyst                              |
|SAP Secur

In [10]:
# drop missing data in salary
# Rows with missing salary values (~57%) were removed to avoid introducing bias through imputation
df = df.dropna(subset=["SALARY"])

In [14]:
df.select("LIGHTCAST_SECTORS_NAME").distinct().show(truncate=False)

+--------------------------------------------------------------------------------------------------------------+
|LIGHTCAST_SECTORS_NAME                                                                                        |
+--------------------------------------------------------------------------------------------------------------+
|[\n  "Green Jobs: Enabling"\n]                                                                                |
|[\n  "Green Jobs: Enabling",\n  "Cybersecurity",\n  "Data Privacy/Protection",\n  "Artificial Intelligence"\n]|
|[\n  "Green Jobs: Enabled"\n]                                                                                 |
|[\n  "Green Jobs: Enabled",\n  "Cybersecurity",\n  "Data Privacy/Protection"\n]                               |
|[\n  "Green Jobs: Enabled",\n  "Data Privacy/Protection"\n]                                                   |
|[\n  "Green Jobs: Enabling",\n  "Data Privacy/Protection",\n  "Artificial Intelligence"\n]     

In [15]:
df.select("NAICS2_NAME").distinct().show(truncate=False)

+------------------------------------------------------------------------+
|NAICS2_NAME                                                             |
+------------------------------------------------------------------------+
|Administrative and Support and Waste Management and Remediation Services|
|Public Administration                                                   |
|Real Estate and Rental and Leasing                                      |
|Information                                                             |
|Unclassified Industry                                                   |
|Accommodation and Food Services                                         |
|Finance and Insurance                                                   |
|Construction                                                            |
|Utilities                                                               |
|Management of Companies and Enterprises                                 |
|Professional, Scientific

In [26]:
df.select("NAICS3_NAME").distinct().show(truncate=False)

+-----------------------------------------------------------------------+
|NAICS3_NAME                                                            |
+-----------------------------------------------------------------------+
|Food Services and Drinking Places                                      |
|Administration of Human Resource Programs                              |
|Primary Metal Manufacturing                                            |
|Textile Product Mills                                                  |
|Support Activities for Agriculture and Forestry                        |
|Credit Intermediation and Related Activities                           |
|Publishing Industries                                                  |
|Textile Mills                                                          |
|Support Activities for Mining                                          |
|Funds, Trusts, and Other Financial Vehicles                            |
|Furniture, Home Furnishings, Electron

**Cleaning the MIN_YEARS_EXPERIENCE column**

Missing values in MIN_YEARS_EXPERIENCE were addressed using a hybrid approach. Internship postings were assigned zero experience based on domain knowledge, while remaining missing values were imputed using the median to preserve the distribution and reduce bias.

In [31]:
# Internship postings are assingned zero experience
from pyspark.sql.functions import when, col
df = df.withColumn(
    "MIN_YEARS_EXPERIENCE_CLEANED",
    when(
        (col("MIN_YEARS_EXPERIENCE_CLEANED").isNull()) & (col("IS_INTERNSHIP") == True),
        0
    ).otherwise(col("MIN_YEARS_EXPERIENCE_CLEANED"))
)
# drop the helper column AFTER using it
df = df.drop("IS_INTERNSHIP")

# check remaining nulls
print("Remaining nulls:",
      df.filter(col("MIN_YEARS_EXPERIENCE_CLEANED").isNull()).count())

Remaining nulls: 21625


In [32]:
# remaining missing values are imputed using the standard method of median 
from pyspark.sql.functions import col

median_exp = df.approxQuantile("MIN_YEARS_EXPERIENCE_CLEANED", [0.5], 0.01)[0]

df= df.fillna({"MIN_YEARS_EXPERIENCE_CLEANED": median_exp})

# Verify that null values are all treated after imputation
df.filter(col("MIN_YEARS_EXPERIENCE_CLEANED").isNull()).count()

0

**Cleaning EMPLOYMENT_TYPE_NAME column**

The EMPLOYMENT_TYPE_NAME variable was cleaned and consolidated into a new feature, EMPLOYMENT_TYPE_NAME_CLEANED, by standardizing and grouping its distinct values into three clear categories: Part-time, Flexible, and Full-time, improving consistency and interpretability for modeling.

In [33]:
# Inspecting EMPLOYMENT_TYPE_NAME for distinct values
df.select("EMPLOYMENT_TYPE_NAME").distinct().show(100, truncate=False)

+------------------------+
|EMPLOYMENT_TYPE_NAME    |
+------------------------+
|Part-time / full-time   |
|Part-time (â‰¤ 32 hours)|
|Full-time (> 32 hours)  |
|NULL                    |
+------------------------+



In [34]:
from pyspark.sql.functions import when, col

# rename column categories 
df = df.withColumn(
    "EMPLOYMENT_TYPE_NAME_CLEANED",
    when(col("EMPLOYMENT_TYPE_NAME_CLEANED") == "Full-time (> 32 hours)", "Full-time")
    .when(col("EMPLOYMENT_TYPE_NAME_CLEANED") == "Part-time (â‰¤ 32 hours)", "Part-time")
    .when(col("EMPLOYMENT_TYPE_NAME_CLEANED") == "Part-time / full-time", "Flexible")
    .otherwise("other")
)

# Inspecting EMPLOYMENT_TYPE_NAME for distinct values
df.select("EMPLOYMENT_TYPE_NAME_CLEANED").distinct().show(100, truncate=False)

+---------------------+
|EMPLOYMENT_TYPE_FINAL|
+---------------------+
|Part-time            |
|other                |
|Flexible             |
|Full-time            |
+---------------------+



**Cleaning the REMOTE_TYPE_NAME column**

The REMOTE_TYPE_NAME variable was cleaned and transformed into a new feature, REMOTE_TYPE_NAME_CLEANED, by standardizing and grouping its distinct values into three consistent categories: Remote, Hybrid, and Onsite, improving clarity and making the variable more suitable for modeling. Additionally, any unknown values were assumed to be Hybrid, reflecting common post-COVID-19 pandemic workplace practices where many roles default to a hybrid structure. This improved both the clarity and practical relevance of the variable for modeling.

In [35]:
# Inspecting REMOTE_TYPE_NAME for distinct values
df.select("REMOTE_TYPE_NAME").distinct().show(100, truncate=False)

+----------------+
|REMOTE_TYPE_NAME|
+----------------+
|Remote          |
|[None]          |
|Not Remote      |
|Hybrid Remote   |
|NULL            |
+----------------+



In [38]:
from pyspark.sql.functions import when, col, lower, trim

df= df.withColumn(
    "REMOTE_TYPE_NAME_CLEANED",
    trim(lower(col("REMOTE_TYPE_NAME")))
)
df = df.withColumn(
    "REMOTE_TYPE_NAME_CLEANED",
    when(col("REMOTE_TYPE_NAME_CLEANED") == "remote", "remote")
    .when(col("REMOTE_TYPE_NAME_CLEANED") == "hybrid remote", "hybrid")
    .when(col("REMOTE_TYPE_NAME_CLEANED") == "not remote", "onsite")
    .when(col("REMOTE_TYPE_NAME_CLEANED") == "[none]", "hybrid")
    .when(col("REMOTE_TYPE_NAME_CLEANED").isNull(), "hybrid")
    .otherwise("other")
)

#verify categories
df.groupBy("REMOTE_TYPE_NAME_CLEANED").count().show()

+-------------------+-----+
|REMOTE_TYPE_CLEANED|count|
+-------------------+-----+
|             onsite| 1127|
|             hybrid|58874|
|             remote|12497|
+-------------------+-----+

